# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and process the FAIR⁲ dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = json.loads(dataset.metadata.to_json())
print(f"Dataset Title: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")
print(f"Published: {metadata_json.get('datePublished')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets (by @id) in the dataset
record_sets = list(dataset.record_sets)
print('Record Sets in dataset:')
for rs in record_sets:
    details = dataset.get_record_set(rs)
    print(f"- @id: {details['@id']}")
    print(f"  Name: {details.get('name')}")
    print(f"  Description: {details.get('description')}")
    # List available fields in this record set
    fields = details.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields/Columns:")
    for f in fields:
        fdict = dataset.get_field(f['@id'])
        print(f"    - @id: {fdict['@id']} -- {fdict.get('name')}: {fdict.get('description')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Pick a record set. For demonstration, we use the first one.
if len(record_sets) == 0:
    raise ValueError("No record sets were found in the dataset metadata.")

selected_record_set_id = record_sets[0]
print(f"Selected record set: {selected_record_set_id}")

# Load records for all available record sets (in real use, you might filter for just a few you need)
dataframes = {}
for record_set_id in record_sets:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}")

df = dataframes[selected_record_set_id]
print("\nColumns in selected record set:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Determine a suitable numeric field (by @id) -- here we try to auto-detect
numeric_col = None
for col in df.columns:
    try:
        # See if column can be cast to float
        df[col].astype(float)
        numeric_col = col
        break
    except:
        continue
if numeric_col is None:
    raise ValueError("No numeric fields detected in this record set.")
print(f"Selected numeric field (by @id): {numeric_col}")

# Convert column to float and filter for demonstration
NUMERIC_THRESHOLD = 10.0
df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')
filtered_df = df[df[numeric_col] > NUMERIC_THRESHOLD]
print(f"Filtered records with {numeric_col} > {NUMERIC_THRESHOLD}:")
print(filtered_df.head())

filtered_df[f"{numeric_col}_normalized"] = (
    filtered_df[numeric_col] - filtered_df[numeric_col].mean()
) / filtered_df[numeric_col].std()
print(f"Normalized {numeric_col} for filtered records:")
print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

# Try grouping by a categorical/text field
group_field = None
for col in df.columns:
    if col != numeric_col and df[col].dtype == object:
        group_field = col
        break
if group_field:
    grouped = filtered_df.groupby(group_field)[numeric_col].mean().reset_index()
    print(f"Grouped mean {numeric_col} by {group_field}:")
    print(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Numeric distribution
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_col].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_col}')
plt.xlabel(numeric_col)
plt.ylabel('Count')
plt.show()

# If group_field, show boxplot
if group_field:
    plt.figure(figsize=(12,5))
    top_gb = df[group_field].value_counts().index[:10]  # for readability, only plot top 10 values
    sns.boxplot(data=df[df[group_field].isin(top_gb)], x=group_field, y=numeric_col)
    plt.title(f'{numeric_col} grouped by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR⁲ dataset, explored its structure, and performed basic data extraction and preprocessing using `mlcroissant`. We demonstrated how to reference record sets and fields by their `@id` fields, filter and normalize numeric fields, and visualize distributions and group differences. This workflow can be extended for deeper domain-specific protein or mass spectrometry data analysis.